# Automated Web Security Header Scanner
**IBM SkillsBuild Cyber Security 2026 — BharatCares**

This notebook checks common HTTP security headers for one URL you are authorized to assess. Missing headers are review findings, not automatic proof of vulnerabilities.

**Student Name:** Sanskruti Dakhale  
**Institution:** Sandip University  
**Year:** Second Year  
**Submission Date:** 23 October 2026


## 1. Install dependency
If running locally, install dependencies with `pip install -r requirements.txt`. In a fresh notebook environment, run the next cell.

In [ ]:
%pip -q install requests

In [ ]:
"""
Automated Web Security Header Scanner
Educational project for IBM SkillsBuild Cyber Security 2026.

Use only on websites you own or have explicit permission to assess.
This tool performs lightweight, non-invasive HTTP checks. It does not exploit vulnerabilities.
"""
from urllib.parse import urlparse
import requests

SECURITY_HEADERS = {
    "Content-Security-Policy": "Helps reduce certain cross-site scripting and content-injection risks.",
    "Strict-Transport-Security": "Tells browsers to use HTTPS for future requests.",
    "X-Content-Type-Options": "Helps prevent MIME-type sniffing.",
    "X-Frame-Options": "Helps mitigate clickjacking in supporting browsers.",
    "Referrer-Policy": "Controls how much referrer information browsers send.",
    "Permissions-Policy": "Controls access to selected browser features."
}

def normalize_url(url):
    url = url.strip()
    if not url:
        raise ValueError("Please enter a URL.")
    if "://" not in url:
        url = "https://" + url
    parsed = urlparse(url)
    if parsed.scheme not in ("http", "https") or not parsed.hostname:
        raise ValueError("Enter a valid http:// or https:// URL.")
    return url

def scan_website(url, timeout=10):
    """Fetch one page and report common security headers and basic CORS behavior."""
    url = normalize_url(url)
    response = requests.get(
        url,
        timeout=timeout,
        allow_redirects=True,
        headers={"User-Agent": "Educational-Web-Security-Header-Scanner/1.0"}
    )
    headers = response.headers
    results = []
    for header, explanation in SECURITY_HEADERS.items():
        value = headers.get(header)
        results.append({
            "header": header,
            "status": "Present" if value else "Missing",
            "value": value or "",
            "explanation": explanation
        })

    cors = headers.get("Access-Control-Allow-Origin", "")
    if cors == "*":
        cors_note = (
            "Wildcard Access-Control-Allow-Origin observed. This is not automatically a vulnerability; "
            "review whether the endpoint exposes sensitive data and whether credentials are allowed."
        )
    elif cors:
        cors_note = f"CORS header present: {cors}. Confirm the allowed origin is intentional."
    else:
        cors_note = "No Access-Control-Allow-Origin header observed in this response."

    return {
        "requested_url": url,
        "final_url": response.url,
        "status_code": response.status_code,
        "https": response.url.lower().startswith("https://"),
        "headers": results,
        "cors": cors_note,
        "server_header": headers.get("Server", ""),
        "content_type": headers.get("Content-Type", "")
    }

def print_report(report):
    print("=" * 72)
    print("AUTOMATED WEB SECURITY HEADER SCANNER")
    print("=" * 72)
    print(f"Requested URL : {report['requested_url']}")
    print(f"Final URL     : {report['final_url']}")
    print(f"HTTP status   : {report['status_code']}")
    print(f"HTTPS         : {'Yes' if report['https'] else 'No'}")
    print("\nSecurity headers:")
    for item in report["headers"]:
        print(f"- {item['header']}: {item['status']}")
        if item["value"]:
            print(f"  Value: {item['value']}")
        print(f"  Note: {item['explanation']}")
    print("\nCORS review:")
    print(report["cors"])
    print("\nImportant: missing headers are review findings, not proof of a vulnerability.")
    print("Verify results manually and consider the application's context.")

if __name__ == "__main__":
    target = input("Enter a website URL you own or are authorized to test: ")
    try:
        result = scan_website(target)
        print_report(result)
    except requests.RequestException as exc:
        print(f"Request failed: {exc}")
    except ValueError as exc:
        print(f"Input error: {exc}")


## 2. Scan an authorized target
Change the URL below to a website you own or have explicit permission to test.

In [ ]:
target_url = "https://example.com"  # Replace only with an authorized target
try:
    report = scan_website(target_url)
    print_report(report)
except requests.RequestException as exc:
    print(f"Request failed: {exc}")
except ValueError as exc:
    print(f"Input error: {exc}")
